In [ ]:
from pathlib import Path
import sys
destination=Path('/kaggle/working/inference_cli_code');destination.mkdir(exist_ok=True)
sources={'predict.py': '"""Run the exported research classifier on an image. No training or downloads."""\nimport argparse\nimport json\nimport math\nfrom pathlib import Path\n\n\ndef load_config(directory):\n    directory = Path(directory).resolve()\n    config = json.loads((directory / \'inference_config.json\').read_text())\n    if config.get(\'architecture\') not in {\'efficientnet_b0\', \'densenet121\', \'resnet18\'}:\n        raise ValueError(\'Unsupported architecture\')\n    if config.get(\'classes\') != [\'NORMAL\', \'PNEUMONIA\']:\n        raise ValueError(\'Unexpected class mapping\')\n    if config.get(\'image_size\', 224) not in {224, 320}:\n        raise ValueError(\'Unsupported image size\')\n    threshold = config.get(\'threshold\')\n    if not isinstance(threshold, (int, float)) or not 0 <= threshold <= 1:\n        raise ValueError(\'Invalid threshold\')\n    names = config.get(\'selected_checkpoints\', [])\n    temperature = config.get(\'temperature\', 1.0)\n    if not isinstance(temperature, (int, float)) or not math.isfinite(temperature) or temperature <= 0:\n        raise ValueError(\'Invalid temperature\')\n    if not names or len(names) != len(set(names)):\n        raise ValueError(\'Missing or repeated checkpoints\')\n    paths = []\n    for name in names:\n        path = (directory / name).resolve()\n        if not path.is_relative_to(directory) or not path.is_file():\n            raise ValueError(\'Checkpoint must exist inside the model directory\')\n        paths.append(path)\n    return config, paths\n\n\ndef predict(directory, image_path, device_name=\'cpu\'):\n    import torch\n    from PIL import Image, ImageOps\n    from torchvision import models, transforms\n    config, paths = load_config(directory)\n    device = torch.device(device_name)\n    image_size = config.get(\'image_size\', 224)\n    with Image.open(image_path) as image:\n        image = ImageOps.pad(image.convert(\'RGB\'), (image_size, image_size),\n                             method=Image.Resampling.BICUBIC, color=0)\n    transform = transforms.Compose([transforms.ToTensor(),\n        transforms.Normalize([.485, .456, .406], [.229, .224, .225])])\n    x = transform(image).unsqueeze(0).to(device)\n    probabilities = []\n    with torch.inference_mode():\n        for path in paths:\n            if config[\'architecture\'] == \'efficientnet_b0\':\n                model = models.efficientnet_b0(weights=None)\n                model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 1)\n            elif config[\'architecture\'] == \'densenet121\':\n                model = models.densenet121(weights=None)\n                model.classifier = torch.nn.Linear(model.classifier.in_features, 1)\n            else:\n                model = models.resnet18(weights=None)\n                model.fc = torch.nn.Linear(model.fc.in_features, 1)\n            model.load_state_dict(torch.load(path, map_location=\'cpu\', weights_only=True))\n            model.to(device).eval()\n            probabilities.append(float(torch.sigmoid(model(x).flatten())[0].cpu()))\n            del model\n    probability = sum(probabilities) / len(probabilities)\n    raw_probability = probability\n    if \'temperature\' in config:\n        p = min(1 - 1e-7, max(1e-7, probability))\n        z = (math.log(p) - math.log1p(-p)) / config[\'temperature\']\n        probability = 1 / (1 + math.exp(-z)) if z >= 0 else math.exp(z) / (1 + math.exp(z))\n    return {\'image\': str(image_path), \'pneumonia_score\': probability,\n            \'threshold\': config[\'threshold\'],\n            \'predicted_label\': config[\'classes\'][int(probability >= config[\'threshold\'])],\n            \'member_scores\': probabilities,\n            \'raw_mean_score\': raw_probability,\n            \'temperature\': config.get(\'temperature\', 1.0),\n            \'image_size\': image_size,\n            \'note\': \'Research image classification score, not a calibrated clinical probability or diagnosis.\'}\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--model-dir\', type=Path, required=True)\n    parser.add_argument(\'--image\', type=Path, required=True)\n    parser.add_argument(\'--device\', choices=[\'cpu\', \'cuda\'], default=\'cpu\')\n    args = parser.parse_args()\n    print(json.dumps(predict(args.model_dir, args.image, args.device), indent=2))\n', 'verify_inference_cli.py': '"""Smoke-test the actual exported inference CLI on a deterministic six-image subset."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nimport pandas as pd\nfrom PIL import Image\n\n\ndef verify(model_dir, data_root, source_dir, lock, output):\n    model_dir, data_root, source_dir, output = map(Path, (model_dir, data_root, source_dir, output))\n    config = json.loads((model_dir / \'inference_config.json\').read_text(encoding=\'utf-8\'))\n    if config != lock[\'configuration\']:\n        raise ValueError(\'Selected configuration differs from frozen model lock\')\n    source_hash = hashlib.sha256((source_dir / \'predict.py\').read_text(encoding=\'utf-8\').encode()).hexdigest()\n    if source_hash != lock[\'inference_source_sha256_lf\']:\n        raise ValueError(\'Inference source changed\')\n    for checkpoint in lock[\'checkpoints\']:\n        if hashlib.sha256((model_dir / checkpoint[\'checkpoint\']).read_bytes()).hexdigest() != checkpoint[\'sha256\']:\n            raise ValueError(\'Checkpoint hash differs from model lock\')\n    frame = pd.read_csv(model_dir / \'test_predictions.csv\', float_precision=\'round_trip\')\n    if len(frame) != 618 or frame.sha256.duplicated().any():\n        raise ValueError(\'Unexpected test manifest\')\n    selected = frame.sort_values(\'sha256\').head(6)\n    output.mkdir(parents=True, exist_ok=True)\n    (output / \'selected_hashes.json\').write_text(json.dumps(selected.sha256.tolist()), encoding=\'utf-8\')\n    records = []\n    environment = dict(os.environ, OMP_NUM_THREADS=\'2\', MKL_NUM_THREADS=\'2\')\n    for row in selected.itertuples():\n        image = data_root / row.original_split / config[\'classes\'][int(row.y_true)] / row.filename\n        with Image.open(image) as original:\n            gray = original.convert(\'L\')\n            digest = hashlib.sha256(str(gray.size).encode() + gray.tobytes()).hexdigest()\n        if digest != row.sha256:\n            raise ValueError(\'Image identity mismatch\')\n        command = [sys.executable, str(source_dir / \'predict.py\'), \'--model-dir\', str(model_dir), \'--image\', str(image), \'--device\', \'cpu\']\n        result = subprocess.run(command, capture_output=True, text=True, check=True, timeout=180, env=environment)\n        prediction = json.loads(result.stdout)\n        difference = abs(prediction[\'pneumonia_score\'] - row.probability)\n        changed = prediction[\'predicted_label\'] != config[\'classes\'][int(row.prediction)]\n        records.append(dict(sha256=row.sha256, saved_score=row.probability,\n                            cli_score=prediction[\'pneumonia_score\'], absolute_score_difference=difference,\n                            changed_classification=changed, exit_code=result.returncode))\n        print(\'CLI image completed\', len(records), flush=True)\n    maximum = max(row[\'absolute_score_difference\'] for row in records)\n    changed = sum(row[\'changed_classification\'] for row in records)\n    passed = maximum <= 1e-4 and changed == 0\n    report = dict(status=\'passed\' if passed else \'failed\', images=len(records),\n                  selection=\'First six test images by ascending decoded-pixel SHA-256\',\n                  max_absolute_score_difference=maximum, changed_classifications=changed,\n                  score_tolerance=1e-4, inference_source_sha256_lf=source_hash,\n                  checkpoint_hashes_verified=lock[\'checkpoints\'], records=records,\n                  scope=\'Actual predict.py subprocess smoke test; not full-cohort or external validation\')\n    (output / \'summary.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    print(\'CLI_SMOKE\', json.dumps(report), flush=True)\n    if not passed:\n        raise RuntimeError(\'Inference CLI discrepancy; preserve result and investigate\')\n    return report\n'}
lock={'recorded_utc': '2026-10-21T21:38:59.854431+00:00', 'status': 'Prospective external-evaluation model lock; external inference not performed', 'basis': 'Combined development-only architecture-resolution selection; source test results already inspected', 'training_kaggle_version': 360888746, 'selection_kaggle_version': 360907021, 'configuration': {'image_size': 320, 'architecture': 'efficientnet_b0', 'selected_checkpoints': ['efficientnet_b0_seed_42/best_weights.pt', 'efficientnet_b0_seed_123/best_weights.pt', 'efficientnet_b0_seed_2026/best_weights.pt'], 'aggregation': 'arithmetic mean of sigmoid probabilities, then positive temperature scaling', 'temperature': 1.340557747633612, 'threshold': 0.8317002265352794, 'classes': ['NORMAL', 'PNEUMONIA'], 'preprocessing': 'RGB; bicubic letterbox 320; ToTensor; ImageNet mean/std', 'selection_architecture_mean_auroc': {'densenet121': 0.9977499637706031, 'efficientnet_b0': 0.99908473102533, 'resnet18': 0.9982838706724939}, 'research_status': 'Exploratory; external confirmation required'}, 'checkpoints': [{'checkpoint': 'efficientnet_b0_seed_42/best_weights.pt', 'sha256': '1cf56a0225e9c9ba65e8808d226703895f0ce8cda7a3acfe6f4e106c26cb8005'}, {'checkpoint': 'efficientnet_b0_seed_123/best_weights.pt', 'sha256': '74f455d965587842516241047541e84981af291a5743c74745ec6d1cf8ab00e8'}, {'checkpoint': 'efficientnet_b0_seed_2026/best_weights.pt', 'sha256': '2a8776986c46efcb8a9bdacd2e0f587a8674d90a4b580e1027cbf44b2b6ca7d4'}], 'verification_versions': [360906861, 360912398], 'inference_source': 'predict.py', 'external_protocol': 'external_evaluation_protocol.json', 'access_status': 'Candidate external dataset access and provenance statements remain unresolved; this lock does not authorize access or accept an agreement', 'adaptation': 'No external-label fitting, temperature refit, cutoff change, checkpoint selection or model selection', 'original_224_comparator': 'Preserved separately; not the locked primary external model', 'limits': 'Checkpoint hashes were read from completed in-Kaggle CPU verification; weight archives are not included in GitHub. No clinical validity or external performance claimed.', 'inference_source_sha256_lf': '2ba76d5b03a78cb108fee19cdfb8f1cd89d3a1dcc9bf05b2e68f87283dbdb64a'}
for name,source in sources.items(): (destination/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(destination))
from verify_inference_cli import verify
root=Path('/kaggle/input')
models=[p.parent for p in root.rglob('inference_config.json') if p.parent.name=='pneumonia_resolution320_outputs']
datasets=sorted({p.parent for p in root.rglob('train') if (p/'NORMAL').is_dir() and (p.parent/'test'/'PNEUMONIA').is_dir()},key=lambda p:(len(p.parts),str(p)))
if len(models)!=1 or not datasets:
    raise ValueError('Attach one 320-pixel training output and one original image dataset')
verify(models[0],datasets[0],destination,lock,'/kaggle/working/inference_cli_outputs')
